# Análisis Diferencial: NPCR y UACI

Compara la parte real cifrada de dos imágenes que difieren en un componente de un píxel. `pipeline.cifrar_imagen_completa` usa el intercalado RGB de `Encriptacion/encrypt.ipynb`. Se comparan valores enteros reconstruidos de los bits cifrados, no la vista normalizada del TIFF.


In [1]:
%pip install numpy
%pip install pillow

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from PIL import Image
from pathlib import Path

from pipeline import cifrar_imagen_completa
from io_imagen import filas_a_matrices_rgb

base = Path().resolve().parent
ruta_imagen = base / "Imagenes" / "House.jpg"


## Pipeline completo

La función comparte los pasos del cifrado: llave, matrices de Pauli, separación en capas y autómata sobre bits RGB intercalados. `filas_a_matrices_rgb` recupera los valores enteros cifrados de cada canal.


## Funciones NPCR y UACI

In [3]:
def npcr(c1, c2):
    """Number of Pixels Change Rate (%). 100% = todos los píxeles cambiaron."""
    c1 = np.asarray(c1)
    c2 = np.asarray(c2)
    assert c1.shape == c2.shape, "Las imágenes cifradas deben tener el mismo tamaño"
    D = (c1 != c2).astype(np.float64)
    return 100.0 * D.sum() / D.size

def uaci(c1, c2, valor_max):
    """Unified Average Changing Intensity (%). valor_max = máximo valor
    representable con el número de bits usado en el cifrado (2**num_bits - 1)."""
    c1 = np.asarray(c1, dtype=np.float64)
    c2 = np.asarray(c2, dtype=np.float64)
    assert c1.shape == c2.shape
    return 100.0 * np.sum(np.abs(c1 - c2)) / (c1.size * valor_max)

## Comparar dos posiciones

El primer píxel cambia la primera fila de la que se deriva la llave; el píxel central deja esa fila intacta. Se muestran ambos casos porque miden comportamientos distintos.


In [4]:
img_original = Image.open(str(ruta_imagen)).convert("RGB")
arr_original = np.array(img_original)
cifrado1 = cifrar_imagen_completa(*img_original.split())
canales1 = filas_a_matrices_rgb(cifrado1.filas_RGB_cif, cifrado1.num_bits, cifrado1.w)

for fila, col in [(0, 0), (img_original.height // 2, img_original.width // 2)]:
    arr_mod = arr_original.copy()
    arr_mod[fila, col, 0] = (int(arr_mod[fila, col, 0]) + 1) % 256
    cifrado2 = cifrar_imagen_completa(*Image.fromarray(arr_mod).split())
    canales2 = filas_a_matrices_rgb(cifrado2.filas_RGB_cif, cifrado2.num_bits, cifrado2.w)
    valor_max = 2**max(cifrado1.num_bits, cifrado2.num_bits) - 1

    print(f"\nPíxel modificado: ({fila}, {col}), llave igual: {cifrado1.key == cifrado2.key}, bits: {cifrado1.num_bits}/{cifrado2.num_bits}")
    print(f"{'Canal':<6}{'NPCR (%)':<16}{'UACI (%)':<16}")
    for nombre, c1, c2 in zip("RGB", canales1, canales2):
        print(f"{nombre:<6}{npcr(c1, c2):<16.6f}{uaci(c1, c2, valor_max):<16.6f}")



Píxel modificado: (0, 0), llave igual: False, bits: 11/13
Canal NPCR (%)        UACI (%)        
R     99.989319       38.429777       
G     99.987793       38.480488       
B     99.989319       38.544895       

Píxel modificado: (128, 128), llave igual: True, bits: 11/11
Canal NPCR (%)        UACI (%)        
R     0.004578        0.000083        
G     0.000000        0.000000        
B     0.000000        0.000000        


## Promedio fuera de la primera fila

Se perturban posiciones aleatorias fuera de la primera fila para medir la propagación del cambio cuando la semilla permanece igual.


In [5]:
def analisis_diferencial_promedio(ruta_imagen, n_pruebas=10, semilla=42):
    rng = np.random.default_rng(semilla)
    img_original = Image.open(str(ruta_imagen)).convert("RGB")
    arr_original = np.array(img_original)
    cifrado1 = cifrar_imagen_completa(*img_original.split())
    canales1 = filas_a_matrices_rgb(cifrado1.filas_RGB_cif, cifrado1.num_bits, cifrado1.w)
    resultados = {canal: {"NPCR": [], "UACI": []} for canal in "RGB"}

    for _ in range(n_pruebas):
        arr_mod = arr_original.copy()
        fila = rng.integers(1, img_original.height)
        col = rng.integers(0, img_original.width)
        canal_idx = rng.integers(0, 3)
        arr_mod[fila, col, canal_idx] = (int(arr_mod[fila, col, canal_idx]) + 1) % 256
        cifrado2 = cifrar_imagen_completa(*Image.fromarray(arr_mod).split())
        canales2 = filas_a_matrices_rgb(cifrado2.filas_RGB_cif, cifrado2.num_bits, cifrado2.w)
        valor_max = 2**max(cifrado1.num_bits, cifrado2.num_bits) - 1

        for nombre, c1, c2 in zip("RGB", canales1, canales2):
            resultados[nombre]["NPCR"].append(npcr(c1, c2))
            resultados[nombre]["UACI"].append(uaci(c1, c2, valor_max))

    print(f"Promedio de {n_pruebas} cambios fuera de la primera fila:")
    print(f"{'Canal':<6}{'NPCR (%)':<16}{'UACI (%)':<16}")
    for canal in "RGB":
        print(f"{canal:<6}{np.mean(resultados[canal]['NPCR']):<16.6f}{np.mean(resultados[canal]['UACI']):<16.6f}")

analisis_diferencial_promedio(ruta_imagen, n_pruebas=10)


Promedio de 10 cambios fuera de la primera fila:
Canal NPCR (%)        UACI (%)        
R     0.000916        0.000011        
G     0.002289        0.000120        
B     0.001373        0.000156        
